In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Training 2-Tier Hierarchical LightGBM Pipeline Layers (`models/train_hierarchical_lightgbm_layers.ipynb`)

This notebook trains the **2-Tier Hierarchical LightGBM Triage Model Architecture** on 35 predictor features, incorporating **Tunable Downsampling Ratios for Layer 1** and **Tunable Upsampling Ratios for Layer 2 (ESI 5)**:

### Tunable Resampling Parameters
- **`l1_downsample_ratio`**: Multiplier relative to ESI 1 count (e.g. `2.0` = sample 2x Non-ESI 1 rows relative to ESI 1). Set `0` to disable.
- **`l2_esi5_upsample_ratio`**: Multiplier relative to ESI 4 count (e.g. `1.0` = oversample ESI 5 to 100% of ESI 4 count, `1.5` = 150%). Set `0` to disable.

### Evaluation Metrics
Reports **Recall (Sensitivity)**, **Specificity**, **Balanced Accuracy**, **ROC-AUC**, and **Confusion Matrices** for Validation and Test splits in `reports/`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(caret)
  library(dplyr)
  library(ggplot2)
  library(tidyr)
  library(pROC)
  library(lightgbm)
})
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) config_path <- "config/triage_conf.json"
config <- fromJSON(config_path)
# Tunable Resampling Ratios (Read from config or override below)
l1_downsample_ratio  <- if (!is.null(config$resampling$layer1_downsample_ratio)) as.numeric(config$resampling$layer1_downsample_ratio) else 2.0
l2_esi5_upsample_ratio <- if (!is.null(config$resampling$layer2_esi5_upsample_ratio)) as.numeric(config$resampling$layer2_esi5_upsample_ratio) else 1.0
cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:          ", config$path$data_source, "\n")
cat("Target Column:              ", config$classes$target_col, "\n")
cat("Test Size:                  ", config$training$test_size, "\n")
cat("Random State:               ", config$training$random_state, "\n")
cat(sprintf("Layer 1 Downsample Ratio:    %.2fx (relative to ESI 1 count)\n", l1_downsample_ratio))
cat(sprintf("Layer 2 ESI 5 Upsample Ratio: %.2fx (relative to ESI 4 count)\n", l2_esi5_upsample_ratio))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Data & Construct 35 Predictor Features
# ---------------------------------------------------------
set.seed(config$training$random_state)
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) data_file <- paste0("../", data_file)
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
raw_df <- get(data_obj_name, envir = data_env)
target_col_name <- config$classes$target_col
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}
p_last   <- get_vec("pulse_last"); p_max    <- get_vec("pulse_max"); p_min    <- get_vec("pulse_min")
s_last   <- get_vec("sbp_last");   s_max    <- get_vec("sbp_max");   s_min    <- get_vec("sbp_min")
o2_last  <- get_vec("spo2_last");  o2_max   <- get_vec("spo2_max");  o2_min   <- get_vec("spo2_min")
r_last   <- get_vec("resp_last");   r_max    <- get_vec("resp_max");   r_min    <- get_vec("resp_min")
t_hr     <- get_vec("triage_vital_hr"); t_sbp <- get_vec("triage_vital_sbp"); t_o2 <- get_vec("triage_vital_o2"); t_rr <- get_vec("triage_vital_rr")
df_full <- data.frame(
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  triage_vital_hr         = t_hr,
  triage_vital_sbp        = t_sbp,
  triage_vital_rr         = t_rr,
  triage_vital_o2         = t_o2,
  pulse_last              = p_last,
  resp_last               = r_last,
  spo2_last               = o2_last,
  sbp_last                = s_last,
  pulse_min               = p_min,
  resp_min                = r_min,
  spo2_min                = o2_min,
  sbp_min                 = s_min,
  pulse_max               = p_max,
  resp_max                = r_max,
  spo2_max                = o2_max,
  sbp_max                 = s_max,
  hr_mean_to_last         = t_hr - p_last,
  sbp_mean_to_last        = t_sbp - s_last,
  spo2_mean_to_last       = t_o2 - o2_last,
  rr_mean_to_last         = t_rr - r_last,
  hr_range                = p_max - p_min,
  rr_range                = r_max - r_min,
  spo2_range              = o2_max - o2_min,
  sbp_range               = s_max - s_min,
  hr_last_to_min          = p_last - p_min,
  rr_last_to_min          = r_last - r_min,
  spo2_last_to_min        = o2_last - o2_min,
  sbp_last_to_min         = s_last - s_min,
  hr_last_to_max          = p_last - p_max,
  rr_last_to_max          = r_last - r_max,
  spo2_last_to_max        = o2_last - o2_max,
  sbp_last_to_max         = s_last - s_max
)
raw_esi <- as.character(raw_df[[target_col_name]])
df_full$target_col <- factor(raw_esi, levels = c("1", "2", "3", "4", "5"))
df_full <- na.omit(df_full)
test_size <- config$training$test_size
val_size  <- config$training$val_size
in_train_val <- createDataPartition(df_full$target_col, p = 1 - test_size, list = FALSE)
train_val_df <- df_full[in_train_val, ]
test_df      <- df_full[-in_train_val, ]
rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df$target_col, p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]
binary_cols <- c("gender", "cc_breathingdifficulty")
cont_cols   <- setdiff(names(train_df), c(binary_cols, "target_col"))
preproc <- preProcess(train_df[, cont_cols, drop = FALSE], method = c("center", "scale"))
train_scaled <- predict(preproc, train_df)
val_scaled   <- predict(preproc, val_df)
test_scaled  <- predict(preproc, test_df)
feat_names <- setdiff(names(train_scaled), "target_col")
cat(sprintf("Partitions Prepared: Train=%d, Val=%d, Test=%d\n", nrow(train_scaled), nrow(val_scaled), nrow(test_scaled)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Train Layer 1 LightGBM with Tunable Downsampling Ratio
# ---------------------------------------------------------
set.seed(config$training$random_state)
esi1_train <- train_scaled %>% filter(target_col == "1")
non1_train <- train_scaled %>% filter(target_col != "1")
n_esi1 <- nrow(esi1_train)
if (l1_downsample_ratio > 0) {
  target_non1_cnt <- round(n_esi1 * l1_downsample_ratio)
  target_non1_cnt <- min(target_non1_cnt, nrow(non1_train))
  sampled_non1_idx <- sample(1:nrow(non1_train), size = target_non1_cnt, replace = FALSE)
  non1_train_sampled <- non1_train[sampled_non1_idx, ]
  train_l1_ready <- rbind(esi1_train, non1_train_sampled)
  cat(sprintf("Layer 1 Downsampling Applied (Ratio=%.2fx): %d total rows (ESI 1=%d, Non-ESI 1=%d)\n\n",
              l1_downsample_ratio, nrow(train_l1_ready), n_esi1, nrow(non1_train_sampled)))
} else {
  train_l1_ready <- train_scaled
  cat(sprintf("Layer 1 Downsampling Disabled: Using full data (%d rows)\n\n", nrow(train_l1_ready)))
}
y_tr_l1 <- ifelse(train_l1_ready$target_col == "1", 1, 0)
y_vl_l1 <- ifelse(val_scaled$target_col == "1", 1, 0)
y_ts_l1 <- ifelse(test_scaled$target_col == "1", 1, 0)
dtrain_l1 <- lgb.Dataset(data = as.matrix(train_l1_ready[, feat_names]), label = y_tr_l1)
dval_l1   <- lgb.Dataset(data = as.matrix(val_scaled[, feat_names]),       label = y_vl_l1)
params_l1 <- list(
  objective        = "binary",
  metric           = "binary_logloss",
  learning_rate    = 0.05,
  num_leaves       = 31,
  max_depth        = 6,
  feature_fraction = 0.8,
  bagging_fraction = 0.8,
  bagging_freq     = 1
)
lgb_l1_model <- lgb.train(
  params                = params_l1,
  data                  = dtrain_l1,
  nrounds               = 100,
  valids                = list(train = dtrain_l1, val = dval_l1),
  early_stopping_rounds = 10,
  verbose               = 0
)
# Evaluate Layer 1 performance on Holdout Test Set
p_l1_test <- predict(lgb_l1_model, as.matrix(test_scaled[, feat_names]))
auc_l1_test <- as.numeric(pROC::roc(y_ts_l1, p_l1_test)$auc)
pred_fac_l1 <- factor(ifelse(p_l1_test >= 0.5, 1, 0), levels = c(1, 0))
act_fac_l1  <- factor(y_ts_l1, levels = c(1, 0))
cm_l1 <- confusionMatrix(pred_fac_l1, act_fac_l1)
rec_l1  <- cm_l1$byClass["Sensitivity"]
spec_l1 <- cm_l1$byClass["Specificity"]
bal_l1  <- cm_l1$byClass["Balanced Accuracy"]
cat("============================================================\n")
cat(sprintf("   LAYER 1 LIGHTGBM (DOWNSAMPLE RATIO = %.2fx) TEST REPORT\n", l1_downsample_ratio))
cat("============================================================\n")
cat(sprintf("  ESI 1 Sensitivity (Recall) : %.4f\n", rec_l1))
cat(sprintf("  Non-ESI 1 Specificity      : %.4f\n", spec_l1))
cat(sprintf("  Balanced Accuracy          : %.4f\n", bal_l1))
cat(sprintf("  ROC-AUC (Test)             : %.4f\n", auc_l1_test))
cat("============================================================\n\n")
cat("Confusion Matrix (Layer 1 ESI 1 Detector):\n")
print(cm_l1$table)
cat("\n\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Train Layer 2 Direct 4-Class LightGBM with Tunable ESI 5 Upsampling Ratio
# ---------------------------------------------------------
set.seed(config$training$random_state)
# Filter training & validation data WITHOUT ESI 1 rows
train_l2_raw <- train_scaled %>% filter(target_col != "1")
val_l2_df    <- val_scaled   %>% filter(target_col != "1")
test_l2_df   <- test_scaled  %>% filter(target_col != "1")
df_esi2 <- train_l2_raw %>% filter(target_col == "2")
df_esi3 <- train_l2_raw %>% filter(target_col == "3")
df_esi4 <- train_l2_raw %>% filter(target_col == "4")
df_esi5 <- train_l2_raw %>% filter(target_col == "5")
if (l2_esi5_upsample_ratio > 0) {
  target_esi5_cnt <- round(nrow(df_esi4) * l2_esi5_upsample_ratio)
  cat(sprintf("Layer 2 ESI 5 Upsampling Applied (Ratio=%.2fx): ESI 2=%d, ESI 3=%d, ESI 4=%d, ESI 5 (Raw)=%d -> Upsampled to %d\n\n",
              l2_esi5_upsample_ratio, nrow(df_esi2), nrow(df_esi3), nrow(df_esi4), nrow(df_esi5), target_esi5_cnt))
  esi5_over <- df_esi5[sample(1:nrow(df_esi5), size = target_esi5_cnt, replace = TRUE), ]
  train_l2_ready <- rbind(df_esi2, df_esi3, df_esi4, esi5_over)
} else {
  train_l2_ready <- train_l2_raw
  cat(sprintf("Layer 2 ESI 5 Upsampling Disabled: Using raw non-ESI 1 data (%d rows)\n\n", nrow(train_l2_ready)))
}
# Map ESI 2..5 to 0-indexed integers (0=ESI2, 1=ESI3, 2=ESI4, 3=ESI5)
y_tr_l2_4cls <- as.numeric(as.character(train_l2_ready$target_col)) - 2
y_vl_l2_4cls <- as.numeric(as.character(val_l2_df$target_col)) - 2
y_ts_l2_4cls <- as.numeric(as.character(test_l2_df$target_col)) - 2
dtrain_l2 <- lgb.Dataset(data = as.matrix(train_l2_ready[, feat_names]), label = y_tr_l2_4cls)
dval_l2   <- lgb.Dataset(data = as.matrix(val_l2_df[, feat_names]),       label = y_vl_l2_4cls)
params_l2 <- list(
  objective        = "multiclass",
  num_class        = 4,
  metric           = "multi_logloss",
  learning_rate    = 0.05,
  num_leaves       = 31,
  max_depth        = 6,
  feature_fraction = 0.8,
  bagging_fraction = 0.8,
  bagging_freq     = 1
)
lgb_l2_model <- lgb.train(
  params                = params_l2,
  data                  = dtrain_l2,
  nrounds               = 100,
  valids                = list(train = dtrain_l2, val = dval_l2),
  early_stopping_rounds = 10,
  verbose               = 0
)
# Evaluate Layer 2 performance on Holdout Test Set
p_l2_mat <- predict(lgb_l2_model, as.matrix(test_l2_df[, feat_names]))
if (!is.matrix(p_l2_mat)) p_l2_mat <- matrix(p_l2_mat, ncol = 4, byrow = TRUE)
preds_l2_4cls <- apply(p_l2_mat, 1, which.max) - 1
cm_l2 <- confusionMatrix(factor(preds_l2_4cls, levels = 0:3), factor(y_ts_l2_4cls, levels = 0:3))
rec_l2  <- mean(as.numeric(cm_l2$byClass[, "Sensitivity"]), na.rm = TRUE)
spec_l2 <- mean(as.numeric(cm_l2$byClass[, "Specificity"]), na.rm = TRUE)
bal_l2  <- mean(as.numeric(cm_l2$byClass[, "Balanced Accuracy"]), na.rm = TRUE)
cat("============================================================\n")
cat(sprintf("   LAYER 2 DIRECT 4-CLASS LIGHTGBM (ESI 5 UPSAMPLE = %.2fx) REPORT\n", l2_esi5_upsample_ratio))
cat("============================================================\n")
cat(sprintf("  Macro Recall (Sens)     : %.4f\n", rec_l2))
cat(sprintf("  Macro Specificity       : %.4f\n", spec_l2))
cat(sprintf("  Macro Balanced Accuracy : %.4f\n", bal_l2))
cat("============================================================\n\n")
cat("Confusion Matrix (Layer 2 4-Class Specialist: 0=ESI2, 1=ESI3, 2=ESI4, 3=ESI5):\n")
print(cm_l2$table)
cat("\n\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Save Model Artifacts to deploy/
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)
# Save Layer 1 LightGBM Model
saveRDS(list(model = lgb_l1_model, preproc = preproc, is_l1_lgb = TRUE), file = file.path(deploy_dir, "lightgbm_layer1_esi1_model.rds"))
cat("Layer 1 LightGBM (ESI 1 Tunable Downsampled Detector) saved to deploy/lightgbm_layer1_esi1_model.rds\n")
# Save Layer 2 Direct 4-Class LightGBM Model
saveRDS(list(model = lgb_l2_model, preproc = preproc, is_lgb_l2_4class = TRUE), file = file.path(deploy_dir, "rf_esi23_esi45_extreme_model.rds"))
cat("Layer 2 LightGBM (Direct 4-Class Specialist with ESI 5 Tunable Upsample) saved to deploy/rf_esi23_esi45_extreme_model.rds\n")
# Save Layer 1 & 2 Training Summary CSV
reports_dir <- "../reports"
if (!dir.exists(reports_dir)) reports_dir <- "reports"
if (!dir.exists(reports_dir)) dir.create(reports_dir, recursive = TRUE)
l1_l2_report <- data.frame(
  Layer = c("Layer1_ESI1_TunableDownsample", "Layer2_Direct_4Class_ESI5_TunableUpsample"),
  Ratio = c(sprintf("Downsample_%.2fx", l1_downsample_ratio), sprintf("ESI5_Upsample_%.2fx", l2_esi5_upsample_ratio)),
  Recall = c(round(rec_l1, 4), round(rec_l2, 4)),
  Specificity = c(round(spec_l1, 4), round(spec_l2, 4)),
  Balanced_Accuracy = c(round(bal_l1, 4), round(bal_l2, 4)),
  ROC_AUC = c(round(auc_l1_test, 4), NA)
)
write.csv(l1_l2_report, file = file.path(reports_dir, "hierarchical_l1_l2_training_report.csv"), row.names = FALSE)
cat("Training report written to reports/hierarchical_l1_l2_training_report.csv\n")